In [ ]:
import numpy as np
import polars as pl


data = pl.read_parquet("../data/1m/1m.parquet").lazy()

In [ ]:
pairs = ["BTCUSDT", "ETHUSDT"]

In [ ]:
def backtest_pairs_trading(
    data: pl.LazyFrame,
    symbol1: str,
    symbol2: str,
    start_date,
    end_date,
    entry_z: float = 2.5,
    fee_bps: float = 10,
    window_size: int = 120,
    print_metrics: bool = True,
):
    """
    Backtest a pairs trading strategy on two symbols.

    Parameters:
    -----------
    data : polars.LazyFrame
        Price data with columns: symbol, open_time, close
    symbol1 : str
        First symbol (e.g., 'BTCUSDT')
    symbol2 : str
        Second symbol (e.g., 'ETHUSDT')
    start_date : datetime
        Start date for backtest
    end_date : datetime
        End date for backtest
    entry_z : float
        Z-score threshold for entry (default: 2.5)
    fee_bps : float
        Fee in basis points per leg (default: 0.2)
    window_size : int
        Rolling window size for calculations (default: 120)
    print_metrics : bool
        Whether to print performance metrics (default: True)

    Returns:
    --------
    dict with keys: 'backtest', 'metrics', 'data'
    """

    fee_rate = fee_bps / 1e4

    # Prepare data
    data_combined = (
        data.filter(pl.col("symbol") == symbol1)
        .select(
            "open_time",
            pl.col("close").alias(f"{symbol1}_price"),
            pl.col("close").log().alias("S1"),
        )
        .join(
            data.filter(pl.col("symbol") == symbol2).select(
                "open_time",
                pl.col("close").alias(f"{symbol2}_price"),
                pl.col("close").log().alias("S2"),
            ),
            on="open_time",
        )
    ).filter((pl.col("open_time") >= start_date) & (pl.col("open_time") < end_date))

    # Backtest
    bt = (
        data_combined.sort("open_time")
        .with_columns(
            pl.rolling_cov("S1", "S2", window_size=window_size).alias("rolling_cov"),
            pl.col("S2").rolling_var(window_size=window_size).alias("rolling_var_s2"),
        )
        .with_columns(
            (pl.col("rolling_cov") / pl.col("rolling_var_s2")).alias("rolling_beta"),
        )
        .with_columns(
            (
                pl.col("S1").rolling_mean(window_size=window_size)
                - pl.col("rolling_beta")
                * pl.col("S2").rolling_mean(window_size=window_size)
            ).alias("rolling_alpha"),
        )
        .with_columns(
            (
                pl.col("S1")
                - (pl.col("rolling_alpha") + pl.col("rolling_beta") * pl.col("S2"))
            ).alias("rolling_residual"),
        )
        .with_columns(
            (
                (
                    pl.col("rolling_residual")
                    - pl.col("rolling_residual").rolling_mean(window_size=window_size)
                )
                / pl.col("rolling_residual").rolling_std(window_size=window_size)
            ).alias("rolling_zscore"),
        )
        .with_columns(
            pl.col(f"{symbol1}_price").pct_change().alias("r_s1"),
            pl.col(f"{symbol2}_price").pct_change().alias("r_s2"),
            pl.col("rolling_beta").shift(1).alias("beta_lag"),
            pl.col("rolling_zscore").shift(1).alias("z_lag"),
        )
        .filter(pl.col("r_s2").is_not_null())
        .with_columns(
            pl.when(pl.col("z_lag") >= entry_z)
            .then(-1)
            .when(pl.col("z_lag") <= -entry_z)
            .then(1)
            .otherwise(0)
            .alias("pos")
        )
        .with_columns(
            (pl.col("r_s1") - pl.col("beta_lag") * pl.col("r_s2")).alias("r_spread_raw"),
            pl.col("pos").shift(1).fill_null(0).alias("pos_prev"),
        )
        .with_columns(
            # Capital-weighted PnL: normalize by (1 + |beta|) to account for unequal capital allocation
            (
                pl.col("pos") * pl.col("r_spread_raw") / (1 + pl.col("beta_lag").abs())
            ).alias("pnl_gross"),
            (
                (pl.col("pos") - pl.col("pos_prev")).abs()
                * (1 + pl.col("beta_lag").abs())
            ).alias("turnover_units"),
        )
        .with_columns((fee_rate * pl.col("turnover_units")).alias("fees"))
        .with_columns((pl.col("pnl_gross") - pl.col("fees")).alias("pnl_net"))
        .with_columns(
            (1 + pl.col("pnl_net")).cum_prod().alias("equity"),
            pl.col("pnl_net").cum_sum().alias("cum_return"),
        )
        .drop_nulls()
    )

    out = bt.collect()

    # Calculate metrics
    ann_factor = 365 * 24 * 60
    mean_pnl = out["pnl_net"].mean()
    std_pnl = out["pnl_net"].std()
    sharpe_ratio = (
        (mean_pnl / std_pnl) * np.sqrt(ann_factor)
        if std_pnl and std_pnl > 0
        else float("nan")
    )
    trades_count = int((out["pos"] != out["pos_prev"]).sum() // 2)

    equity_arr = out["equity"].to_numpy()
    roll_max = np.maximum.accumulate(equity_arr)
    dd = (equity_arr / roll_max) - 1.0
    max_drawdown = dd.min()

    metrics = {
        "symbol1": symbol1,
        "symbol2": symbol2,
        "bars": len(out),
        "trades": trades_count,
        "mean_minute_pnl": float(mean_pnl),
        "std_minute_pnl": float(std_pnl),
        "sharpe_annualized": float(sharpe_ratio),
        "max_drawdown": float(max_drawdown),
        "final_equity": float(equity_arr[-1]) if len(equity_arr) else None,
        "total_return": float(out["cum_return"][-1]) if len(out) else None,
    }

    if print_metrics:
        print(metrics)

    return {"backtest": out, "metrics": metrics, "data": data_combined.collect()}

In [ ]:
from datetime import datetime
from itertools import product

# Define parameter ranges
entry_z_range = [2.0]
window_size_range = [30]

# Define date range for backtest
start_date = datetime(2024, 1, 1)
end_date = datetime(2024, 12, 31)

# Store results
results = []

# Grid search over all parameter combinations
for symbol1, symbol2 in [
    (pairs[i], pairs[j]) for i in range(len(pairs)) for j in range(i + 1, len(pairs))
]:
    for entry_z, window_size in product(
        entry_z_range, window_size_range
    ):
        try:
            result = backtest_pairs_trading(
                data,
                symbol1=symbol1,
                symbol2=symbol2,
                start_date=start_date,
                end_date=end_date,
                entry_z=entry_z,
                window_size=window_size,
                print_metrics=False,
                fee_bps=0.0,
            )

            metrics = result["metrics"]
            metrics["entry_z"] = entry_z
            metrics["window_size"] = window_size
            results.append(metrics)
        except Exception as e:
            pass

# Convert to DataFrame and sort by Sharpe ratio
results_df = pl.DataFrame(results).sort("sharpe_annualized", descending=True)
results_df.head(10)

In [ ]:
def backtest_single_asset(
    data: pl.LazyFrame,
    symbol: str,
    start_date,
    end_date,
    entry_z: float = 2.0,
    fee_bps: float = 10,
    window_size: int = 120,
    print_metrics: bool = True,
):
    """
    Backtest a mean reversion strategy on a single asset with 1-minute hold period.
    
    Parameters:
    -----------
    data : polars.LazyFrame
        Price data with columns: symbol, open_time, close
    symbol : str
        Symbol to trade (e.g., 'ETHUSDT')
    start_date : datetime
        Start date for backtest
    end_date : datetime
        End date for backtest
    entry_z : float
        Z-score threshold for entry (default: 2.0)
    fee_bps : float
        Fee in basis points (default: 10)
    window_size : int
        Rolling window size for calculations (default: 120)
    print_metrics : bool
        Whether to print performance metrics (default: True)
    
    Returns:
    --------
    dict with keys: 'backtest', 'metrics'
    """
    
    fee_rate = fee_bps / 1e4
    
    # Prepare data
    asset_data = (
        data.filter(pl.col("symbol") == symbol)
        .select(
            "open_time",
            pl.col("close").alias("price"),
            pl.col("close").log().alias("log_price"),
        )
        .filter((pl.col("open_time") >= start_date) & (pl.col("open_time") < end_date))
        .sort("open_time")
    )
    
    # Backtest
    bt = (
        asset_data
        .with_columns(
            pl.col("log_price").rolling_mean(window_size=window_size).alias("rolling_mean"),
            pl.col("log_price").rolling_std(window_size=window_size).alias("rolling_std"),
        )
        .with_columns(
            (
                (pl.col("log_price") - pl.col("rolling_mean")) / pl.col("rolling_std")
            ).alias("zscore")
        )
        .with_columns(
            pl.col("price").pct_change().alias("returns"),
            pl.col("zscore").shift(1).alias("zscore_lag"),
        )
        .filter(pl.col("returns").is_not_null())
        .with_columns(
            # Mean reversion: buy when zscore is low (oversold), sell when high (overbought)
            pl.when(pl.col("zscore_lag") <= -entry_z)
            .then(1)  # Long when oversold
            .when(pl.col("zscore_lag") >= entry_z)
            .then(-1)  # Short when overbought
            .otherwise(0)
            .alias("pos")
        )
        .with_columns(
            pl.col("pos").shift(1).fill_null(0).alias("pos_prev"),
        )
        .with_columns(
            (pl.col("pos") * pl.col("returns")).alias("pnl_gross"),
            (pl.col("pos") - pl.col("pos_prev")).abs().alias("turnover"),
        )
        .with_columns((fee_rate * pl.col("turnover")).alias("fees"))
        .with_columns((pl.col("pnl_gross") - pl.col("fees")).alias("pnl_net"))
        .with_columns(
            (1 + pl.col("pnl_net")).cum_prod().alias("equity"),
            pl.col("pnl_net").cum_sum().alias("cum_return"),
        )
        .drop_nulls()
    )
    
    out = bt.collect()
    
    # Calculate metrics
    ann_factor = 365 * 24 * 60
    mean_pnl = out["pnl_net"].mean()
    std_pnl = out["pnl_net"].std()
    sharpe_ratio = (
        (mean_pnl / std_pnl) * np.sqrt(ann_factor)
        if std_pnl and std_pnl > 0
        else float("nan")
    )
    trades_count = int((out["pos"] != out["pos_prev"]).sum() // 2)
    
    equity_arr = out["equity"].to_numpy()
    roll_max = np.maximum.accumulate(equity_arr)
    dd = (equity_arr / roll_max) - 1.0
    max_drawdown = dd.min()
    
    metrics = {
        "symbol": symbol,
        "bars": len(out),
        "trades": trades_count,
        "mean_minute_pnl": float(mean_pnl),
        "std_minute_pnl": float(std_pnl),
        "sharpe_annualized": float(sharpe_ratio),
        "max_drawdown": float(max_drawdown),
        "final_equity": float(equity_arr[-1]) if len(equity_arr) else None,
        "total_return": float(out["cum_return"][-1]) if len(out) else None,
    }
    
    if print_metrics:
        print(metrics)
    
    return {"backtest": out, "metrics": metrics}

In [ ]:
# Test pure ETH strategy
result_eth = backtest_single_asset(
    data,
    symbol="ETHUSDT",
    start_date=datetime(2024, 1, 1),
    end_date=datetime(2024, 12, 31),
    entry_z=2.0,
    window_size=15,
    fee_bps=0,
    print_metrics=True,
)

In [ ]:
# Grid search for single asset strategy
entry_z_range = [2.0]
window_size_range = [30]

results_single = []

for entry_z in entry_z_range:
    for window_size in window_size_range:
        try:
            result = backtest_single_asset(
                data,
                symbol="ETHUSDT",
                start_date=datetime(2024, 1, 1),
                end_date=datetime(2024, 12, 31),
                entry_z=entry_z,
                window_size=window_size,
                fee_bps=0.2,
                print_metrics=False,
            )
            
            metrics = result["metrics"]
            metrics["entry_z"] = entry_z
            metrics["window_size"] = window_size
            results_single.append(metrics)
        except Exception as e:
            print(f"Error with entry_z={entry_z}, window_size={window_size}: {e}")

# Convert to DataFrame and sort by Sharpe ratio
results_single_df = pl.DataFrame(results_single).sort("sharpe_annualized", descending=True)
results_single_df

In [ ]:
# Test pairs trading with capital weighting fix
result_pairs = backtest_pairs_trading(
    data,
    symbol1="BTCUSDT",
    symbol2="ETHUSDT",
    start_date=datetime(2024, 1, 1),
    end_date=datetime(2024, 12, 31),
    entry_z=2.0,
    window_size=30,
    fee_bps=0.1,
    print_metrics=True,
)

# Check average beta to understand capital allocation
bt_data = result_pairs["backtest"]
avg_beta = bt_data["beta_lag"].abs().mean()
print(f"\n平均 |beta|: {avg_beta:.4f}")
print(f"平均资金配置 - BTC: {1/(1+avg_beta):.2%}, ETH: {avg_beta/(1+avg_beta):.2%}")